[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/dlt-certified/blob/main/notebooks/day-09-advanced.ipynb)

---
# Day 9 · Advanced Patterns: REST API Helper and Normalisation
**certified-journeys / dlt-certified** &nbsp;|&nbsp; Advanced

> **Goal for today:** Use the `rest_api` source helper to build complex multi-endpoint pipelines in configuration only, chain dependent resources, and understand how dlt automatically flattens nested JSON into relational tables.

---
## The `rest_api` source helper

Writing a `@dlt.resource` for every API endpoint is repetitive. The `rest_api` helper lets you describe an entire API as a configuration dictionary — no custom code for pagination, auth, or endpoint wiring.

| Approach | Lines of code | Handles pagination | Handles auth | Handles chaining |
|---|---|---|---|---|
| Custom `@dlt.resource` | 30–80 per endpoint | Manual | Manual | Manual |
| `rest_api` config | 10–20 per endpoint | Automatic | Automatic | Declarative |

> **Tip:** The `rest_api` helper can build 80% of API sources with zero custom code — use it before writing from scratch.

### Config structure overview

```python
source = rest_api({
    "client": {"base_url": "...", "auth": {...}},
    "resource_defaults": {...},  # shared settings for all resources
    "resources": [
        {"name": "users", "endpoint": {"path": "/users"}},
        # child resource — depends on parent
        {
            "name": "posts",
            "endpoint": {
                "path": "/posts",
                "params": {"userId": {"type": "resolve", "resource": "users", "field": "id"}}
            }
        },
    ]
})
```

---
## Step 1 · Install dlt with REST API support

In [ ]:
%pip install -q "dlt[duckdb]" requests

---
## Step 2 · Basic `rest_api` source — JSONPlaceholder API

[JSONPlaceholder](https://jsonplaceholder.typicode.com/) is a free, public REST API for testing. It has users, posts, comments, and more — no auth required.

In [ ]:
import dlt
from dlt.sources.rest_api import rest_api_source

# Minimal rest_api config — fetches /users and /posts in parallel
jsonplaceholder_source = rest_api_source(
    {
        "client": {
            "base_url": "https://jsonplaceholder.typicode.com",
            # No auth needed for JSONPlaceholder
            # For APIs with auth, you'd add:
            # "auth": {"type": "bearer", "token": dlt.secrets["sources.my_api.token"]}
        },
        "resource_defaults": {
            "write_disposition": "replace",
            "primary_key": "id",
        },
        "resources": [
            # Simple endpoint — just fetch all users
            {
                "name": "users",
                "endpoint": {
                    "path": "/users",
                    # JSONPlaceholder returns all items at once (no pagination needed)
                    # For paginated APIs, add: "paginator": {"type": "page_number", ...}
                }
            },
            # Fetch all posts (flat endpoint)
            {
                "name": "posts",
                "endpoint": {
                    "path": "/posts",
                }
            },
            # Fetch all comments
            {
                "name": "comments",
                "endpoint": {
                    "path": "/comments",
                }
            },
        ]
    }
)

pipeline = dlt.pipeline(
    pipeline_name="jsonplaceholder",
    destination="duckdb",
    dataset_name="jph",
)

info = pipeline.run(jsonplaceholder_source)
print(info)

In [ ]:
# Inspect what landed
with pipeline.sql_client() as client:
    for table in ["jph.users", "jph.posts", "jph.comments"]:
        with client.execute_query(f"SELECT COUNT(*) as cnt FROM {table}") as cur:
            cnt = cur.df().iloc[0]["cnt"]
            print(f"{table}: {cnt} rows")

    # Show what the users table looks like
    with client.execute_query("SELECT id, name, email, username FROM jph.users LIMIT 3") as cur:
        print("\nSample users:")
        print(cur.df().to_string(index=False))

**What just happened?**
- 3 endpoints loaded with a single config dict — no custom resource functions
- dlt handled HTTP requests, response parsing, and schema inference automatically
- JSONPlaceholder returns 10 users, 100 posts, 500 comments — all landed correctly

---
## Step 3 · Endpoint chaining — posts per user

Endpoint chaining lets a child resource depend on parent records. For each user, fetch their posts at `/posts?userId={id}`.

The `"resolve"` param type tells dlt: *for each record from the `users` resource, call this endpoint with the user's `id`.*

In [ ]:
from dlt.sources.rest_api import rest_api_source

# Chained source: users → posts per user → comments per post
chained_source = rest_api_source(
    {
        "client": {
            "base_url": "https://jsonplaceholder.typicode.com",
        },
        "resource_defaults": {
            "write_disposition": "replace",
            "primary_key": "id",
        },
        "resources": [
            # ── Level 1: Users (root resource) ────────────────────────────
            {
                "name": "users",
                "endpoint": {"path": "/users"},
            },

            # ── Level 2: Posts per user (child of users) ──────────────────
            # For each user, fetch: GET /posts?userId={user.id}
            {
                "name": "user_posts",
                "endpoint": {
                    "path": "/posts",
                    "params": {
                        # 'resolve' means: take the value of 'id' from each 'users' record
                        "userId": {
                            "type": "resolve",
                            "resource": "users",
                            "field": "id",
                        }
                    }
                },
                # Explicit dependency declaration
                "include_from_parent": ["id"],  # include parent user's id in each post record
            },

            # ── Level 3: Comments per post (child of user_posts) ──────────
            # For each post, fetch: GET /comments?postId={post.id}
            {
                "name": "post_comments",
                "endpoint": {
                    "path": "/comments",
                    "params": {
                        "postId": {
                            "type": "resolve",
                            "resource": "user_posts",
                            "field": "id",
                        }
                    }
                },
            },
        ]
    }
)

chained_pipeline = dlt.pipeline(
    pipeline_name="jph_chained",
    destination="duckdb",
    dataset_name="chained",
)

# This makes multiple HTTP requests: 1 for users, 10 for posts (one per user),
# and up to 100 for comments (one per post). Be patient.
print("Loading chained endpoints... (this makes ~111 HTTP requests)")
info = chained_pipeline.run(chained_source)
print(info)

In [ ]:
with chained_pipeline.sql_client() as client:
    for table in ["chained.users", "chained.user_posts", "chained.post_comments"]:
        with client.execute_query(f"SELECT COUNT(*) as cnt FROM {table}") as cur:
            cnt = cur.df().iloc[0]["cnt"]
            print(f"{table}: {cnt} rows")

    # Verify chaining: each post should have a userId linking back to users
    with client.execute_query(
        "SELECT user_id, COUNT(*) as post_count FROM chained.user_posts GROUP BY user_id ORDER BY user_id"
    ) as cur:
        print("\nPosts per user:")
        print(cur.df().to_string(index=False))

**What just happened?**
- dlt executed the chain: load all users → for each user, load their posts → for each post, load comments
- The `resolve` param automatically substituted `{user.id}` into the URL
- `include_from_parent` added the parent's `id` to each child record — you get a natural foreign key
- 3 tables, fully linked, from one config dict

---
## Step 4 · Nested JSON normalisation

Real API responses contain nested objects and arrays. dlt flattens these automatically — nested objects become prefixed columns, nested arrays become separate tables with foreign keys.

In [ ]:
import dlt

# Deeply nested data — simulates a real API response with nested objects and arrays
@dlt.resource(primary_key="user_id", write_disposition="replace")
def nested_users():
    yield [
        {
            "user_id": 1,
            "name": "Alice",
            # Nested object → flattened as address__street, address__city, etc.
            "address": {
                "street": "123 Main St",
                "city": "Anytown",
                "geo": {"lat": "40.7128", "lng": "-74.0060"},  # double-nested
            },
            # Nested array → becomes a separate table: nested_users__tags
            "tags": ["vip", "premium", "early-adopter"],
            # Array of objects → becomes a separate table: nested_users__orders
            "recent_orders": [
                {"order_id": "O001", "amount": 99.99,  "status": "completed"},
                {"order_id": "O002", "amount": 49.99,  "status": "pending"},
            ],
            "company": {"name": "Acme Corp", "bs": "synergise real-time e-markets"},
        },
        {
            "user_id": 2,
            "name": "Bob",
            "address": {
                "street": "456 Oak Ave",
                "city": "Othertown",
                "geo": {"lat": "34.0522", "lng": "-118.2437"},
            },
            "tags": ["standard"],
            "recent_orders": [
                {"order_id": "O003", "amount": 199.99, "status": "shipped"},
            ],
            "company": {"name": "Globex", "bs": "harness cross-platform experiences"},
        },
    ]

nested_pipeline = dlt.pipeline(
    pipeline_name="nested_demo",
    destination="duckdb",
    dataset_name="nested",
)

info = nested_pipeline.run(nested_users())
print(info)

In [ ]:
# See ALL the tables that dlt created from one nested resource
with nested_pipeline.sql_client() as client:
    # DuckDB: list tables in the schema
    with client.execute_query(
        "SELECT table_name FROM information_schema.tables WHERE table_schema = 'nested' ORDER BY table_name"
    ) as cur:
        tables = [row[0] for row in cur.fetchall()]
        print("Tables created by dlt:")
        for t in tables:
            print(f"  nested.{t}")

    # The main table — nested objects are flattened with __ separator
    print("\n── Main table: nested.nested_users ──")
    with client.execute_query("SELECT * FROM nested.nested_users") as cur:
        df = cur.df()
        # Show only the columns (many are flattened nested fields)
        print("Columns:", list(df.columns))
        print(df.to_string(index=False))

In [ ]:
# Inspect the child tables — arrays became separate tables with foreign keys
with nested_pipeline.sql_client() as client:
    # Tags array → flat table
    try:
        with client.execute_query("SELECT * FROM nested.nested_users__tags ORDER BY _dlt_parent_id") as cur:
            print("── nested_users__tags (each tag as a row) ──")
            print(cur.df().to_string(index=False))
    except Exception as e:
        print(f"tags table: {e}")

    # Recent orders array → table with FK back to parent
    try:
        with client.execute_query("SELECT * FROM nested.nested_users__recent_orders ORDER BY order_id") as cur:
            print("\n── nested_users__recent_orders (each order as a row) ──")
            print(cur.df().to_string(index=False))
    except Exception as e:
        print(f"recent_orders table: {e}")

**What just happened?**
- dlt split one nested record into multiple tables automatically
- Nested objects (`address`, `address.geo`, `company`) → flattened into columns with `__` separator
- Nested arrays (`tags`, `recent_orders`) → separate tables named `parent_table__array_field`
- Each child table has `_dlt_parent_id` and `_dlt_root_id` — the foreign keys back to the parent

### How normalisation works

```
nested_users                          (main table)
├── address__street                   (flattened object column)
├── address__city                     (flattened object column)
├── address__geo__lat                 (doubly nested object column)
├── address__geo__lng
├── company__name
├── company__bs
├── _dlt_id                           (dlt row identifier)
├── _dlt_load_id
│
nested_users__tags                    (child table: array of scalars)
├── value                             (the scalar value)
├── _dlt_parent_id                    (FK → nested_users._dlt_id)
├── _dlt_root_id
│
nested_users__recent_orders           (child table: array of objects)
├── order_id
├── amount
├── status
├── _dlt_parent_id                    (FK → nested_users._dlt_id)
└── _dlt_root_id
```

---
## Step 5 · Controlling normalisation with `max_table_nesting`

In [ ]:
import dlt
import json

# max_table_nesting=0 → everything is flattened into ONE table
# Arrays are JSON-encoded as strings instead of becoming child tables

@dlt.resource(
    primary_key="user_id",
    write_disposition="replace",
    max_table_nesting=0,  # no child tables — flatten everything
)
def flat_users():
    yield [
        {
            "user_id": 1,
            "name": "Alice",
            "address": {"street": "123 Main St", "city": "Anytown"},
            "tags": ["vip", "premium"],  # will be JSON-encoded string
            "recent_orders": [
                {"order_id": "O001", "amount": 99.99}
            ],
        },
        {
            "user_id": 2,
            "name": "Bob",
            "address": {"street": "456 Oak Ave", "city": "Othertown"},
            "tags": ["standard"],
            "recent_orders": [
                {"order_id": "O002", "amount": 49.99}
            ],
        },
    ]


# max_table_nesting=1 → one level deep, then stop (default is ~3)
@dlt.resource(
    primary_key="user_id",
    write_disposition="replace",
    max_table_nesting=1,  # one child table level allowed
)
def medium_nested_users():
    yield [
        {
            "user_id": 1,
            "name": "Alice",
            "address": {"street": "123 Main St", "city": "Anytown"},
            "tags": ["vip", "premium"],
            "recent_orders": [
                {"order_id": "O001", "amount": 99.99}
            ],
        },
    ]


flat_pipeline = dlt.pipeline(
    pipeline_name="nesting_comparison",
    destination="duckdb",
    dataset_name="nesting",
)

info = flat_pipeline.run([flat_users(), medium_nested_users()])
print(info)

In [ ]:
# Compare: flat_users (nesting=0) vs medium_nested_users (nesting=1)
with flat_pipeline.sql_client() as client:
    # List all tables in this dataset
    with client.execute_query(
        "SELECT table_name FROM information_schema.tables WHERE table_schema = 'nesting' ORDER BY table_name"
    ) as cur:
        all_tables = [row[0] for row in cur.fetchall()]
        print("Tables created:")
        for t in all_tables:
            print(f"  nesting.{t}")

    # flat_users: arrays are JSON strings, no child tables
    print("\n── flat_users (max_table_nesting=0): all in one table ──")
    with client.execute_query("SELECT user_id, name, tags, recent_orders FROM nesting.flat_users") as cur:
        df = cur.df()
        print(df.to_string(index=False))
        print(f"\ntags column type: {df['tags'].dtype}  (JSON string — not split into rows)")

**Normalisation modes at a glance:**

| `max_table_nesting` | Effect | Use when |
|---|---|---|
| `0` | No child tables. Arrays → JSON strings. Objects → flattened columns. | Destination doesn't support JOINs, or you want one flat table |
| `1` | One level of child tables. | Shallow nesting, want mostly flat |
| `2` (default) | Two levels of child tables. | Most real-world APIs |
| `None` / omit | Unlimited nesting. | Deeply nested documents |

---
## Step 6 · Inspect multiple tables from a real nested API response

In [ ]:
from dlt.sources.rest_api import rest_api_source

# JSONPlaceholder /users returns nested objects (address, company, geo)
# Let's see how dlt splits them
users_source = rest_api_source(
    {
        "client": {"base_url": "https://jsonplaceholder.typicode.com"},
        "resource_defaults": {"write_disposition": "replace", "primary_key": "id"},
        "resources": [
            {
                "name": "full_users",
                "endpoint": {"path": "/users"},
                # Default nesting — let dlt split nested objects and arrays
            },
        ]
    }
)

users_pipeline = dlt.pipeline(
    pipeline_name="jph_users",
    destination="duckdb",
    dataset_name="jph_users",
)

info = users_pipeline.run(users_source)
print(info)

with users_pipeline.sql_client() as client:
    with client.execute_query(
        "SELECT table_name FROM information_schema.tables WHERE table_schema = 'jph_users' ORDER BY table_name"
    ) as cur:
        tables = [row[0] for row in cur.fetchall()]
        print(f"\nTables from /users endpoint: {tables}")

    # Main table columns — show how address.* and company.* were flattened
    with client.execute_query("SELECT * FROM jph_users.full_users LIMIT 2") as cur:
        df = cur.df()
        print("\nColumns in main table:")
        for col in df.columns:
            print(f"  {col}")

---
## Challenge — do this yourself

1. Use `rest_api_source` to fetch `/todos` from JSONPlaceholder (returns 200 items with fields: `userId`, `id`, `title`, `completed`)
2. Use endpoint chaining to fetch todos per user: for each user from `/users`, fetch `/todos?userId={id}`
3. After loading, query: how many completed vs pending todos per user?
4. Try setting `max_table_nesting=0` on the todos resource — what changes?

In [ ]:
# Your solution here


<details>
<summary>Show solution</summary>

```python
from dlt.sources.rest_api import rest_api_source

todos_source = rest_api_source({
    "client": {"base_url": "https://jsonplaceholder.typicode.com"},
    "resource_defaults": {"write_disposition": "replace", "primary_key": "id"},
    "resources": [
        {"name": "users", "endpoint": {"path": "/users"}},
        {
            "name": "user_todos",
            "endpoint": {
                "path": "/todos",
                "params": {
                    "userId": {"type": "resolve", "resource": "users", "field": "id"}
                }
            },
        },
    ]
})

p = dlt.pipeline(pipeline_name="todos", destination="duckdb", dataset_name="todos")
p.run(todos_source)

with p.sql_client() as client:
    with client.execute_query("""
        SELECT user_id,
               SUM(CASE WHEN completed THEN 1 ELSE 0 END) AS done,
               SUM(CASE WHEN NOT completed THEN 1 ELSE 0 END) AS pending
        FROM todos.user_todos
        GROUP BY user_id ORDER BY user_id
    """) as cur:
        print(cur.df().to_string(index=False))
```

With `max_table_nesting=0`: `/todos` items are flat by default (no nested objects), so no visible change. But if the response contained arrays, they'd become JSON strings instead of child tables.
</details>

---
## Day 9 key concepts recap

| Concept | What to remember |
|---|---|
| `rest_api_source()` | Config-driven API source — no custom request code |
| `"client": {"base_url": ...}` | Base URL shared across all resources |
| `"resource_defaults"` | Shared settings: `write_disposition`, `primary_key`, `paginator` |
| `"type": "resolve"` | Endpoint chaining — substitutes parent field value into child URL |
| `include_from_parent` | Copies parent fields into child records for natural FKs |
| Nested objects | Flattened into columns: `address.city` → `address__city` |
| Nested arrays | Split into child tables: `tags` → `resource__tags` with `_dlt_parent_id` |
| `max_table_nesting=0` | Everything in one table — arrays become JSON strings |
| `_dlt_parent_id` | Foreign key from child table back to parent row |

> **Tip:** If you find yourself writing `requests.get()` loops in a resource, check if `rest_api_source` can handle it first. It supports OAuth, header auth, cursor pagination, page-number pagination, and offset pagination out of the box.

---
## What's next → Day 10

**Day 10** → Capstone project: build a complete production pipeline combining all concepts from Days 1–9, document it, and publish to GitHub.

Mark Day 9 complete in your [tracker](../index.html).